# 02. UGV 로봇 상태 확인 (State)

영상 대응: **21:59–23:25**

> 3차 교육 영상 20–30분의 기능을 재구성한 **로컬 실습본**입니다. 공식 원본 노트북이 아닙니다. `mil_msgs` 원본 정의가 없어 이 환경에서는 `sim_mil_interfaces`를 사용합니다. 실제 ROS 2로 시뮬레이터와 통신합니다.

`/mil/state`와 `/mil/match`를 구독해 체력, 배터리, 현재 모드와 경기 진행 상태를 확인합니다.

| 항목 | 영상 설명 |
|---|---|
| 로봇 상태 갱신 | 2Hz, 초당 2회 |
| 경기 상태 갱신 | 상태 변경 시 |
| 예제 관측 시간 | 10초, 변경 가능 |
| 주요 필드 | `hp`, `hp_max`, `battery_pct`, `mode`, `cmd` |

| 모드 | 체력·탄약 | 주행·조준·발사 |
|---|---|---|
| practice | 자동 회복 | 경기 단계 잠금 없음. 피격 잠금·재발사 대기는 적용 |
| mission | 자동 회복 없음 | 경기 진행 중 허용 |

`idle`, `standby`, `pause`, `report`, `stop`은 경기 제어 잠금 상태이며 `start`, `resume`은 진행 상태입니다. 아래 `control`은 수신한 모드와 상태를 이용해 코드에서 계산한 값입니다. 피격 잠금과 재발사 판정은 명령 응답도 확인해야 합니다.


## 실행 방법

위쪽 커널을 **Python 3 (ROS 2 · simulation)** 으로 선택하고, 아래 셀을 위에서부터 **Shift+Enter**로 실행합니다. 물리 경기장과 ROS 실습 연결을 먼저 켜세요. 각 실습은 시간이 지나면 종료되며, 셀을 다시 실행할 수 있습니다.


In [ ]:
import time
import math
import uuid
import rclpy
from rclpy.qos import QoSProfile, ReliabilityPolicy, DurabilityPolicy, qos_profile_sensor_data
from rclpy.signals import SignalHandlerOptions
from sim_mil_interfaces.msg import State as MissionState, Match as MatchCommand

# 상태/경기 토픽은 마지막 값을 새 구독자에게도 전달합니다.
STATUS_QOS = QoSProfile(depth=1, reliability=ReliabilityPolicy.RELIABLE,
                        durability=DurabilityPolicy.TRANSIENT_LOCAL)

def new_node(prefix):
    if not rclpy.ok():
        # Jupyter의 중단 버튼이 KeyboardInterrupt와 finally 정지를 처리하게 둡니다.
        rclpy.init(args=[], signal_handler_options=SignalHandlerOptions.NO)
    return rclpy.create_node(prefix + '_' + uuid.uuid4().hex[:8])

def spin_until(node, predicate, timeout=5.0):
    deadline = time.monotonic() + timeout
    while not predicate() and time.monotonic() < deadline:
        rclpy.spin_once(node, timeout_sec=min(0.05, max(0.0, deadline-time.monotonic())))
    if not predicate():
        raise TimeoutError('ROS 데이터/응답 시간 초과. 물리 경기장과 ROS 연결 상태를 확인하세요.')

def watch_status(node):
    data = {'state': None, 'match': None, 'state_at': None}
    def on_state(msg):
        data['state'] = msg
        data['state_at'] = time.monotonic()
    def on_match(msg):
        data['match'] = msg
    subscriptions = [
        node.create_subscription(MissionState, '/mil/state', on_state, STATUS_QOS),
        node.create_subscription(MatchCommand, '/mil/match', on_match, STATUS_QOS),
    ]
    spin_until(node, lambda: data['state'] is not None and data['match'] is not None)
    return data, subscriptions

def allowed(data):
    if data['state_at'] is None or time.monotonic()-data['state_at'] > 1.5:
        return False
    match = data['match']
    if match is None or match.cmd == 'pause':
        return False
    return match.mode == 'practice' or (match.mode == 'mission' and match.cmd in ('start', 'resume'))

def require_control(data):
    if not allowed(data):
        raise RuntimeError('제어 잠금 또는 상태 데이터 지연. 경기 재개·연결 상태를 확인하세요.')

def wait_publisher(node, publisher, timeout=5.0):
    spin_until(node, lambda: publisher.get_subscription_count() > 0, timeout)

def yaw_degrees(q):
    return math.degrees(math.atan2(2.0*(q.w*q.z + q.x*q.y), 1.0-2.0*(q.y*q.y+q.z*q.z)))

def source_age_seconds(node, message, maximum, label):
    stamp = message.header.stamp
    source_ns = int(stamp.sec)*1_000_000_000 + int(stamp.nanosec)
    if source_ns <= 0:
        raise ValueError(label + ': 원본 센서 시각이 없습니다.')
    age = (node.get_clock().now().nanoseconds-source_ns)/1_000_000_000
    if age < -0.25:
        raise ValueError(f'{label}: 센서 시각이 {-age:.3f}초 미래입니다. ROS 시계 설정을 확인하세요.')
    if age > maximum:
        raise TimeoutError(f'{label}: 원본 데이터가 {age:.3f}초 전 값입니다.')
    return age


## 실습 설정

아래 값을 바꾸고 다시 실행할 수 있습니다.


In [ ]:
MONITOR_SECONDS = 10.0
STALE_SECONDS = 1.5


## 실행


In [ ]:
node = new_node('demo_state')
DEMO_RESULT = {'demo':'02', 'samples':0}
try:
    data, subscriptions = watch_status(node)
    end = time.monotonic() + max(0.1, min(float(MONITOR_SECONDS), 60.0))
    next_print = 0.0
    while time.monotonic() < end:
        rclpy.spin_once(node, timeout_sec=0.05)
        now = time.monotonic()
        if now < next_print:
            continue
        next_print = now + 0.5
        state, match = data['state'], data['match']
        age = now - data['state_at']
        if age > STALE_SECONDS:
            print(f'상태 지연: 마지막 수신 {age:.1f}초 전 · Locked')
            DEMO_RESULT.update(stale=True, age_seconds=age)
            continue
        control = 'Allowed' if allowed(data) else 'Locked'
        print(f'HP {state.hp}/{state.hp_max} | BAT {state.battery_pct:.0f}% | {match.mode} | {match.cmd} | {control}')
        DEMO_RESULT.update(samples=DEMO_RESULT['samples']+1, hp=int(state.hp), hp_max=int(state.hp_max),
                           battery_pct=float(state.battery_pct), mode=match.mode, cmd=match.cmd, control=control, stale=False)
finally:
    node.destroy_node()


[← 실습 목차](00.실습목차.ipynb) · [코드 편집](../team_node.py)

실물 장치의 토픽 타입·주기·응답과 맞는지는 지급 UGV에서 확인해야 합니다.
